# Web Scraping

Chapter 17 asked a server for data and the server handed it over as JSON, already structured, in the shape its designers intended. This chapter is about the other case: the data is on a web page, meant for a person to read, and nobody built you a door.

**Scraping is reading a page that was written for a human and pulling the data out of it.** It is genuinely useful and it is the least stable thing in this course, because you are depending on the arrangement of somebody else's HTML, which they can change on a Tuesday without telling you.

That is why this chapter starts with two questions that come before any code, and ends with the ones about whether you should be doing it at all.

**What we will learn:**

1. Whether to scrape at all, and whether you are allowed to
2. How a page is put together
3. Why chapter 16's regex could not do this, shown on a real page
4. BeautifulSoup: `find`, `find_all`, attributes and text
5. CSS selectors, which are usually the shorter way
6. Pulling a table out
7. Writing a scraper that survives the page changing
8. Following pagination with a chapter 12 generator
9. Saving the result to CSV, from chapter 10
10. One real request, and how to make it politely

**About the pages in this notebook.** Nearly everything here parses HTML files committed in `sample_data/`, so the output is the same every time and works offline. Three cells at the end share a single real request, to a site that exists specifically so people can practise on it. They are marked.

---
# 1. Before You Write Any Code

### Is there an API?

Chapter 17's whole point. An API gives you structured data, a stated contract, and permission. Scraping gives you none of those.

| | API (ch. 17) | Scraping |
|---|---|---|
| You get | JSON, already structured | HTML meant for a person |
| Stability | changes are versioned and announced | changes whenever a designer feels like it |
| Permission | explicit, in the terms | you have to go and check |
| Speed | one request, exactly the fields you asked for | a whole page of markup for three numbers |
| Breaks when | they retire a version | they move a `<div>` |

Look for an API before you write a scraper. Try `example.com/api`, search "*site name* API", read the developer page, and open the browser's network tab while using the site: a page that loads data as you scroll is usually calling a JSON endpoint you can call directly.

### Are you allowed to?

Three separate questions, and they have different answers.

**`robots.txt`** is a file at the root of a site saying which paths automated clients should leave alone. It is a request, not a lock, but ignoring it is the clearest possible statement that you knew and did not care. Python reads it with a standard library module:

In [1]:
from urllib.robotparser import RobotFileParser

ROBOTS = """
User-agent: *
Crawl-delay: 2
Disallow: /cart/
Disallow: /checkout/
Disallow: /search?
Allow: /

User-agent: GreedyBot
Disallow: /
"""

rules = RobotFileParser()
rules.parse(ROBOTS.splitlines())            # parse text we already have

for path in ["/books/", "/books/page-2.html", "/cart/", "/checkout/pay"]:
    allowed = rules.can_fetch("*", "https://example.shop" + path)
    print(f"  {path:<22} {'allowed' if allowed else 'DISALLOWED'}")

  /books/                allowed
  /books/page-2.html     allowed
  /cart/                 DISALLOWED
  /checkout/pay          DISALLOWED


Two more things that file is telling you:

In [2]:
print("crawl delay asked of everyone:", rules.crawl_delay("*"), "seconds")
print("GreedyBot may fetch the home page:", rules.can_fetch("GreedyBot", "https://example.shop/"))

crawl delay asked of everyone: 2 seconds
GreedyBot may fetch the home page: False


`Crawl-delay: 2` is the site asking for one request every two seconds. `GreedyBot` has been named and banned outright, which is what happens to a scraper that behaves badly.

In a real script you fetch the live file instead of pasting it in:

```python
rules = RobotFileParser()
rules.set_url("https://example.shop/robots.txt")
rules.read()
```

A missing `robots.txt` returns a 404 and the parser treats everything as allowed. That means no rules were stated, not that anything goes.

**The terms of service** are the legally meaningful document, and they are separate from `robots.txt`. Many sites forbid automated collection outright. Read them.

**Personal data** is its own question again. Names, emails, photographs and profiles are covered by laws such as the GDPR wherever the person lives, regardless of the data being publicly visible. "It was on a public page" is not a defence, and this is the part that turns a scraping project into a legal problem.

The rest of this chapter assumes you have answered all three and the answer was yes.

---
# 2. How a Page Is Put Together

HTML is a tree of **elements**. Each one has a tag name, optional **attributes**, and content that can be text or more elements.

```html
<article class="product" data-sku="9781617294433">
  <h3><a href="/book/9781617294433.html">Deep Learning with Python</a></h3>
  <p class="price">£51.77</p>
</article>
```

- `article`, `h3`, `a`, `p` are tags.
- `class`, `data-sku`, `href` are attributes. `class` and `id` are the ones you will target most, because that is what the site's own styling uses.
- The `<a>` is inside the `<h3>`, which is inside the `<article>`. That nesting is the tree you will navigate.

`sample_data/` has three pages of a small bookshop, written for this chapter. Here is the beginning of the first one:

In [3]:
from pathlib import Path

page1 = Path("sample_data/shop_page_1.html").read_text(encoding="utf-8")

print(page1[:640])

<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <title>The Bookshop - page 1</title>
</head>
<body>
  <header>
    <h1>The Bookshop</h1>
    <p class="tagline">Books for people who write code</p>
  </header>

  <!-- <article class="product" data-sku="0000000000000">
         <h3><a href="/book/retired.html">Retired Title</a></h3>
         <p class="price">&pound;99.99</p>
       </article> -->

  <main id="catalogue">
    <article class="product" data-sku="9781617294433">
      <h3><a href="/book/9781617294433.html">Deep Learning with Python</a></h3>
      <p class="author">François Chollet</p>
      <p class="pri


Notice the comment block. Those four lines are invisible on the page: a browser skips them, and so should anything reading the page. Remember them.

---
# 3. Why Chapter 16's Regex Cannot Do This

Chapter 16 ended by saying a regex should not parse HTML, and showed it failing on a three-tag example. Here it is on a realistic page. The pattern looks exactly right:

In [4]:
import re

print(re.findall(r'<article class="product" data-sku="(\d+)">', page1))

['0000000000000', '9781617294433', '9781492056355']


Three results, and the page has three products, so it looks like it worked.

It did not. Compare with what a parser finds:

In [5]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(page1, "html.parser")

print([article["data-sku"] for article in soup.find_all("article", class_="product")])

['9781617294433', '9781593279288', '9781492056355']


Different lists, of the same length, which is the worst kind of wrong.

- `0000000000000` is the retired product inside the HTML comment. It is not on the page. The regex sees text; it has no idea what a comment is.
- `9781593279288` is a real product that the regex missed, because its `data-sku` sits on the next line and the pattern expected a space.

Both mistakes are silent, and a report built on that list would be wrong in two directions at once. A parser reads the page the way a browser does: comments are comments, whitespace inside a tag is whitespace, and the result is a tree you can ask questions of.

`"html.parser"` is the one built into Python, and it is what this chapter uses. `"lxml"` is faster and needs installing; for a page you are reading once, the difference does not matter.

---
# 4. Finding Things

Two methods do most of the work.

- `find(...)` returns the **first** match, or `None`.
- `find_all(...)` returns a **list** of every match, possibly empty.

In [6]:
first = soup.find("article", class_="product")

print(type(first).__name__)
print(first.h3.get_text(strip=True))

Tag
Deep Learning with Python


The argument is `class_`, with an underscore. `class` is a Python keyword, so it cannot be a parameter name.

Reading a tag gives you three things: its text, its attributes, and the tags inside it.

In [7]:
print("text of the whole card:")
print(" ", first.get_text(" ", strip=True))
print()
print("just the title  :", first.h3.a.get_text(strip=True))
print("the link        :", first.h3.a["href"])
print("the sku         :", first["data-sku"])
print("the price       :", first.find("p", class_="price").text)

text of the whole card:
  Deep Learning with Python François Chollet £51.77 Rating In stock (22 available)

just the title  : Deep Learning with Python
the link        : /book/9781617294433.html
the sku         : 9781617294433
the price       : £51.77


`first.h3.a` is shorthand: reading an attribute that is not a real attribute gives you the first child tag with that name. It is convenient and it is fragile, because it returns `None` the moment the page changes, so keep it for exploring rather than for code you rely on.

### Three ways to get text, and the one that surprises people

| | |
|---|---|
| `.text` | all text inside, including from nested tags |
| `.get_text(strip=True)` | the same, with whitespace tidied |
| `.string` | the text **only if there is exactly one child** |

In [8]:
title_tag = first.h3

print("h3 .text  :", repr(title_tag.text))
print("h3 .string:", repr(title_tag.string))
print()
print("article .text  :", repr(first.text[:40]), "...")
print("article .string:", repr(first.string))

h3 .text  : 'Deep Learning with Python'
h3 .string: 'Deep Learning with Python'

article .text  : '\nDeep Learning with Python\nFrançois Chol' ...
article .string: None


`.string` on the `<article>` is `None`, because that tag has several children and there is no single string to return. It is not an error, and a script that used it would carry on with `None`. Use `.get_text()` and forget `.string` exists.

### Attributes are a dictionary, except `class`

In [9]:
rating = first.find("p", class_="rating")

print("all attributes:", rating.attrs)
print("class         :", rating["class"], "<- a list, not a string")
print("second class  :", rating["class"][1])

all attributes: {'class': ['rating', 'star-four']}
class         : ['rating', 'star-four'] <- a list, not a string
second class  : star-four


`class` comes back as a **list**, because an element can have several classes. Every other attribute is a plain string. Writing `tag["class"] == "rating"` therefore never matches, which is a confusing five minutes the first time.

### Finding many, and narrowing

In [10]:
for article in soup.find_all("article", class_="product"):
    title = article.h3.get_text(strip=True)
    price = article.find("p", class_="price").text
    stock = article.find("p", class_="stock").get_text(strip=True)
    print(f"  {title:<28} {price:>8}   {stock}")

  Deep Learning with Python      £51.77   In stock (22 available)
  Python Crash Course            £23.99   In stock (8 available)
  Fluent Python                  £44.50   Out of stock


---
# 5. CSS Selectors

`select()` and `select_one()` take a CSS selector, the same language the site's own stylesheet uses. For anything nested, they are much shorter than chained `find` calls.

| Selector | Matches |
|---|---|
| `p` | every `<p>` |
| `.price` | anything with `class="price"` |
| `#catalogue` | the element with `id="catalogue"` |
| `article.product` | `<article>` elements that have `class="product"` |
| `article p.price` | a `.price` anywhere inside an `<article>` |
| `article > h3` | an `<h3>` that is a **direct** child |
| `a[href]` | `<a>` elements that have an `href` |
| `p.stock.in-stock` | both classes at once |

In [11]:
print("prices        :", [p.text for p in soup.select("article.product p.price")])
print("in stock      :", len(soup.select("p.stock.in-stock")), "of",
                          len(soup.select("article.product")))
print("first title   :", soup.select_one("#catalogue h3 a").get_text(strip=True))
print("the next link :", soup.select_one("nav.pagination a.next")["href"])

prices        : ['£51.77', '£23.99', '£44.50']
in stock      : 2 of 3
first title   : Deep Learning with Python
the next link : shop_page_2.html


A tip that saves a lot of guessing: in your browser, right-click the thing you want, choose Inspect, then right-click the highlighted element and copy its selector. Trim what you get down to the part that identifies the element rather than its exact position, because the copied version is usually far too specific to survive any change to the page.

---
# 6. Pulling Out a Table

Tables are the friendliest thing on any page, because the structure already matches what you want: `<tr>` is a row, `<th>` is a heading, `<td>` is a cell.

In [12]:
table = soup.find("table", class_="specs")

headers = [th.get_text(strip=True) for th in table.select("thead th")]
rows = [[td.get_text(strip=True) for td in tr.find_all("td")]
        for tr in table.select("tbody tr")]

print(headers)
for row in rows:
    print(" ", row)

['Method', 'Days', 'Cost']
  ['Standard', '5', '£2.99']
  ['Express', '2', '£6.49']
  ['Next day', '1', '£9.99']


That is already the shape chapter 10's `csv.writer` wants. If you reach for pandas later, `pandas.read_html` does this whole cell in one line, which is worth knowing about but is a topic for the repository that covers pandas.

---
# 7. Writing a Scraper That Does Not Fall Over

The page will change. Sooner or later a field is missing, and the difference between a scraper that reports "3 of 4 rows had no price" and one that dies at 3am is a few lines.

Page 2 has a product with no price, which is exactly the sort of thing a real catalogue does:

In [13]:
page2 = Path("sample_data/shop_page_2.html").read_text(encoding="utf-8")
soup2 = BeautifulSoup(page2, "html.parser")

article = soup2.select("article.product")[1]
print("found the price tag:", article.find("p", class_="price"))

found the price tag: None


`find` returned `None`, which is fine until the next line:

In [14]:
try:
    print(article.find("p", class_="price").text)
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: 'NoneType' object has no attribute 'text'


`'NoneType' object has no attribute 'text'` is the single most common scraping error, and it is the same shape as chapter 16's `re.search` returning `None`.

Attributes fail differently. A missing one raises `KeyError` with `[...]` and returns `None` with `.get()`, exactly like a dictionary in chapter 3:

In [15]:
link = article.find("a")

try:
    link["data-discount"]
except KeyError as e:
    print("KeyError:", e)

print('with .get():', link.get("data-discount"))
print('with a default:', link.get("data-discount", "none"))

KeyError: 'data-discount'
with .get(): None
with a default: none


So a parsing function that will survive contact with a real site looks like this. One helper, and every field goes through it:

In [16]:
def text_of(parent, selector, default=None):
    # The first match for `selector` as tidy text, or `default` if it is not there.
    found = parent.select_one(selector)
    return found.get_text(strip=True) if found else default


def parse_product(article):
    return {
        "sku": article.get("data-sku"),
        "title": text_of(article, "h3 a"),
        "author": text_of(article, "p.author"),
        "price": text_of(article, "p.price", default=""),
        "stock": text_of(article, "p.stock", default="unknown"),
    }


for article in soup2.select("article.product"):
    print(parse_product(article))

{'sku': '9781593279929', 'title': 'Automate the Boring Stuff', 'author': 'Al Sweigart', 'price': '£27.95', 'stock': 'In stock (13 available)'}
{'sku': '9781098139482', 'title': 'Effective Pandas', 'author': 'Matt Harrison', 'price': '', 'stock': 'In stock (3 available)'}
{'sku': '9781492041139', 'title': 'Data Science from Scratch', 'author': 'Joel Grus', 'price': '£41.05', 'stock': 'In stock (11 available)'}


No exception, and the gap is visible in the output rather than hidden. Turning the price into a number is chapter 16's job:

In [17]:
def to_price(text):
    digits = re.sub(r"[^\d.]", "", text or "")      # chapter 16
    return float(digits) if digits else None


for article in soup2.select("article.product"):
    product = parse_product(article)
    print(f"  {product['title']:<28} {to_price(product['price'])}")

  Automate the Boring Stuff    27.95
  Effective Pandas             None
  Data Science from Scratch    41.05


---
# 8. Following the Pages

Chapter 17 paginated by asking for page 1, 2, 3 until one came back empty. A website usually tells you instead: there is a "next" link, until there is not.

That is a generator again, and the stopping condition is now "no next link".

In [18]:
def crawl(start_page, load):
    # Yield the soup for each page, following the 'next' link until it runs out.
    page = start_page
    while page:
        html = load(page)
        soup = BeautifulSoup(html, "html.parser")
        print(f"  (parsed {page})")
        yield soup

        next_link = soup.select_one("nav.pagination a.next")
        page = next_link["href"] if next_link else None

In [19]:
def load_from_samples(name):
    return Path(f"sample_data/{name}").read_text(encoding="utf-8")


products = []
for page_soup in crawl("shop_page_1.html", load_from_samples):
    for article in page_soup.select("article.product"):
        products.append(parse_product(article))

print(f"\n{len(products)} products across the whole catalogue")
for product in products[:4]:
    print(f"  {product['title']:<40} {product['price']}")

  (parsed shop_page_1.html)
  (parsed shop_page_2.html)
  (parsed shop_page_3.html)

9 products across the whole catalogue
  Deep Learning with Python                £51.77
  Python Crash Course                      £23.99
  Fluent Python                            £44.50
  Automate the Boring Stuff                £27.95


Three pages, nine products, and the crawl stopped by itself because page 3 has no next link.

Because `crawl` is a generator, chapter 15's `islice` still works, and the pages you do not reach are never loaded:

In [20]:
from itertools import islice

for page_soup in islice(crawl("shop_page_1.html", load_from_samples), 2):
    print("   ", page_soup.title.get_text(strip=True))

  (parsed shop_page_1.html)
    The Bookshop - page 1
  (parsed shop_page_2.html)
    The Bookshop - page 2


---
# 9. Saving the Result

The point of scraping is the file at the end of it. Chapter 10's `csv.DictWriter` takes the list of dictionaries directly.

In [21]:
import csv
import os

os.makedirs("scraper_demo", exist_ok=True)
out_path = "scraper_demo/books.csv"

fields = ["sku", "title", "author", "price", "stock"]

with open(out_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(products)

print(f"wrote {len(products)} rows to {out_path}\n")
print(Path(out_path).read_text(encoding="utf-8")[:320])

wrote 9 rows to scraper_demo/books.csv

sku,title,author,price,stock
9781617294433,Deep Learning with Python,François Chollet,£51.77,In stock (22 available)
9781593279288,Python Crash Course,Eric Matthes,£23.99,In stock (8 available)
9781492056355,Fluent Python,Luciano Ramalho,£44.50,Out of stock
9781593279929,Automate the Boring Stuff,Al Sweigart,£27.95,In 


Chapter 15's `Counter` will now tell you something about what you collected, which is the natural next step and a good check that the scrape worked:

In [22]:
from collections import Counter

print(Counter(p["stock"].split(" (")[0] for p in products).most_common())
print("cheapest:", min(products, key=lambda p: to_price(p["price"]) or 999)["title"])

[('In stock', 7), ('Out of stock', 2)]
cheapest: Python Crash Course


---
# 10. One Real Request

Everything so far read files from disk. Here is the same code against a live site, using `requests` from chapter 17 to fetch and BeautifulSoup to parse.

The site is `books.toscrape.com`, which exists specifically so people can practise scraping without bothering anyone. This is the only request the notebook makes.

Three things make the request a polite one:

- a real `User-Agent` saying what this is,
- a `timeout`, as chapter 17 insisted,
- a `sleep` afterwards, because the next request would otherwise arrive immediately.

In [23]:
import time
import requests

response = requests.get("https://books.toscrape.com/",
                        headers={"User-Agent": "python-course-notebook/1.0 (learning example)"},
                        timeout=10)
response.raise_for_status()

first_price = BeautifulSoup(response.text, "html.parser").select_one("p.price_color")
print("the first price reads:", first_price.get_text(strip=True))

the first price reads: Â£51.77


That is not a currency symbol anyone recognises. The page is UTF-8, in which `£` is two bytes, and something decoded those two bytes as two separate characters.

The something is `requests`. When a server does not say which encoding it used, the HTTP standard tells the client to assume Latin-1, so that is what `requests` does. This page's server says only `text/html`, with no charset, while the HTML itself declares UTF-8 in a `<meta>` tag that `requests` never looks at.

In [24]:
print("what requests assumed:", response.encoding)
print("what the bytes say    :", response.apparent_encoding)

what requests assumed: ISO-8859-1
what the bytes say    : utf-8


`apparent_encoding` guesses from the bytes rather than the header, and it is right here. Assign it and `response.text` is decoded again, correctly, with no second request:

In [25]:
response.encoding = response.apparent_encoding     # believe the bytes

live = BeautifulSoup(response.text, "html.parser")

for pod in live.select("article.product_pod")[:5]:
    title = pod.h3.a["title"]
    price = pod.select_one("p.price_color").get_text(strip=True)
    stock = pod.select_one("p.instock").get_text(strip=True)
    print(f"  {title[:34]:<34} {price:>8}   {stock}")

time.sleep(1)                               # do not fetch the next page immediately

  A Light in the Attic                 £51.77   In stock
  Tipping the Velvet                   £53.74   In stock
  Soumission                           £50.10   In stock
  Sharp Objects                        £47.82   In stock
  Sapiens: A Brief History of Humank   £54.23   In stock


Mojibake, as those `Â` characters are called, is one of the most common scraping bugs, and it is easy to miss because the script does not fail. It writes a CSV full of `Â£` and everything downstream inherits it.

Two ways to avoid it. Set `response.encoding = "utf-8"` when you know the site, which nearly every site now is. Or hand BeautifulSoup `response.content`, the raw bytes, and let it read the `<meta>` tag itself.

`response.text` rather than `response.json()`, because this time the body is HTML.

Note `pod.h3.a["title"]`. The title is in an attribute rather than in the text, because the visible text is truncated with an ellipsis. That is a small example of the general rule: look at the actual HTML before deciding where the data is.

---
# 11. Scraping Responsibly

The technical part of this chapter is short. This part is what separates a useful tool from a problem.

### Do not hurt the site

A loop with no delay sends hundreds of requests a second, which is indistinguishable from an attack and is treated as one. One request every second or two is plenty for any job that is not urgent, and `robots.txt` may have asked for a specific delay, as ours did.

```python
for url in urls:
    scrape(url)
    time.sleep(2)          # or whatever Crawl-delay asked for
```

Scrape once and save. Re-parsing a file on disk costs the site nothing, which is the reason this notebook has `sample_data/`.

### Say who you are

A real `User-Agent` with a project name and a contact address means that when something goes wrong, the site owner can email you rather than block your whole network. Pretending to be Chrome is the opposite of that, and if you feel the need to disguise your scraper, that feeling is information.

### Take only what you need

Fetching one page and extracting four fields is not the same as copying a database. Ask what you would say if the site owner asked what you were doing.

### The legal part, briefly and honestly

This is not legal advice, and the law varies by country. What is reasonably settled:

- **`robots.txt` is not a law**, but ignoring it undermines any claim that you acted in good faith.
- **Terms of service are a contract**, and many of them prohibit automated collection. Breaking them can end an account and has supported legal claims.
- **Copyright still applies.** Facts are generally not copyrightable; the text, photographs and layout expressing them generally are. Extracting prices is a different act from republishing reviews.
- **Personal data is regulated** by the GDPR and similar laws almost everywhere, and public visibility is not consent.
- **Bypassing a login, a paywall or a CAPTCHA** moves you into computer-misuse territory in most jurisdictions. This chapter does not cover any of that, deliberately.

The short version: read `robots.txt`, read the terms, go slowly, identify yourself, take the minimum, avoid personal data, and prefer the API. If a project needs you to ignore several of those, that is the project telling you something.

---
# 12. Common Mistakes

| Mistake | What happens | Fix |
|---|---|---|
| Scraping when an API exists | Slower, more fragile, possibly not allowed | Look for the API first |
| A regex instead of a parser | Silently finds comments and misses real elements | `BeautifulSoup` |
| `.find(...).text` with no check | `'NoneType' object has no attribute 'text'` | Check for `None`, or use a helper |
| `tag["href"]` on a missing attribute | `KeyError` | `tag.get("href")` |
| `tag["class"] == "price"` | Never true; `class` is a list | `"price" in tag["class"]`, or `select` |
| `.string` on a tag with children | `None`, and no error | `.get_text(strip=True)` |
| A loop with no `sleep` | You look like an attack, and get blocked | One request every second or two |
| A fake browser `User-Agent` | Nobody can contact you; you get blocked anyway | Name your project and give an address |
| Copied browser selectors | Break on any layout change | Target `class` and `id`, not position |
| Re-fetching while developing | Wasted requests on the same page | Save the HTML once, then work offline |

---
# 13. Summary: Your Scraping Cheat Sheet

**The whole shape**

```python
import requests, time
from bs4 import BeautifulSoup

response = requests.get(url, headers={"User-Agent": "my-project/1.0 (me@example.com)"},
                        timeout=10)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")
...
time.sleep(2)
```

**Finding**

| | |
|---|---|
| `soup.find("p")` | first `<p>`, or `None` |
| `soup.find("p", class_="price")` | first with that class |
| `soup.find_all("article", class_="product")` | every match, as a list |
| `soup.select("article.product p.price")` | CSS selector, a list |
| `soup.select_one("#catalogue h3 a")` | CSS selector, first match |

**Reading**

| | |
|---|---|
| `tag.get_text(strip=True)` | the text, tidied |
| `tag.text` | the text, as it is |
| `tag["href"]` | an attribute, `KeyError` if missing |
| `tag.get("href")` | an attribute, `None` if missing |
| `tag["class"]` | a **list** of classes |
| `tag.attrs` | every attribute, as a dict |

**Selectors**

| | |
|---|---|
| `.price` | by class |
| `#catalogue` | by id |
| `article.product` | tag and class together |
| `article p.price` | anywhere inside |
| `article > h3` | direct child only |
| `a[href]` | has that attribute |

**Not falling over**

```python
def text_of(parent, selector, default=None):
    found = parent.select_one(selector)
    return found.get_text(strip=True) if found else default
```

**Checking you are allowed**

```python
from urllib.robotparser import RobotFileParser

rules = RobotFileParser()
rules.set_url("https://example.com/robots.txt")
rules.read()
rules.can_fetch("*", url)      # True or False
rules.crawl_delay("*")         # seconds, or None
```

**Before you start**

1. Is there an API?
2. What does `robots.txt` say?
3. What do the terms of service say?
4. Is there personal data in this?
5. How slowly can I afford to go?

---

**Next:** chapter 19 is about making code you can trust: type hints, docstrings, the `logging` module, and tests with `pytest`. Everything in chapters 17 and 18 depends on somebody else's server behaving, which makes them exactly the code that most needs testing.